# Importação de Bibliotecas

In [ ]:
from google.colab import drive

import os
import gc
from datetime import datetime
from math import trunc

import numpy as np
import pandas as pd


from tqdm.notebook import tqdm



# Abertura do Arquivo

In [ ]:

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DIRETORIO_BASE = "."

# Definição do grupos e modelos

In [ ]:
grupos = ["intents", "permissions", "opcodes", "apicalls", "permissions_opcodes", "todas"]

# Definição das features a serem utilizadas

In [ ]:
for nome_grupo in tqdm(grupos):
  caminho_raiz = f"{DIRETORIO_BASE}/src_shap/{nome_grupo}/"

  caminho_mlp = os.path.join(caminho_raiz, "mlp_limiar5/")
  caminho_rf = os.path.join(caminho_raiz, "rf/")
  caminho_xgb = os.path.join(caminho_raiz, "xgb/")

  arquivo_features_importantes_mlp = os.path.join(caminho_mlp, f"{nome_grupo}__mlp__ranking_global_SHAP.csv")
  arquivo_features_importantes_rf = os.path.join(caminho_rf, f"{nome_grupo}__rf__ranking_global_SHAP.csv")
  arquivo_features_importantes_xgb = os.path.join(caminho_xgb, f"{nome_grupo}__xgb__ranking_global_SHAP.csv")


  df_mlp = pd.read_csv(arquivo_features_importantes_mlp)
  df_rf = pd.read_csv(arquivo_features_importantes_rf)
  df_xgb = pd.read_csv(arquivo_features_importantes_xgb)


  df_merged = df_mlp.merge(df_rf, on="feature", suffixes=("_mlp", "_rf"))
  df_merged = df_merged.merge(df_xgb, on="feature")
  df_merged = df_merged.rename(columns={"importance_mean_abs_shap": "importance_mean_abs_shap_xgb"})

  col_mlp = "importance_mean_abs_shap_mlp"
  col_rf = "importance_mean_abs_shap_rf"
  col_xgb = "importance_mean_abs_shap_xgb"

  for col in [col_mlp, col_rf, col_xgb]:
      soma = df_merged[col].sum()
      df_merged[col + "_norm"] = df_merged[col] / soma if soma > 0 else 0.0


  df_merged["importance_media"] = df_merged[
      [col_mlp + "_norm", col_rf + "_norm", col_xgb + "_norm"]
  ].mean(axis=1)


  df_merged = df_merged.sort_values(by="importance_media", ascending=False)
  df_merged.to_csv(os.path.join(caminho_raiz, "features_importancias_medias_geral.csv"), index=False)


  df_final = df_merged[df_merged["importance_media"] > 0]
  df_final = df_final.sort_values(by="importance_media", ascending=False)
  df_final.to_csv(os.path.join(caminho_raiz, "features_importancias_medias_nao_zero.csv"), index=False)
